In [ ]:
!pip install -r requirements.txt
from warpdrive import WarpDrive
import requests
import urllib.parse
import uuid
import time
import random
from faker import Faker
from datetime import datetime, timedelta
import json
from json import dumps
 
 
fake = Faker()
# Sample prompt templates and topics
templates = [
    "Explain {topic} like I'm five.",
    "Write a short story about {topic}.",
    "Give an example of {topic} in real life.",
    "What's the opposite of {topic}?",
    "Summarize a debate about {topic}.",
    "How does {topic} affect our daily life?",
    "What are the pros and cons of {topic}?",
    "What is the future of {topic}?",
    "Why is {topic} important in school?",
    "Describe a funny situation involving {topic}.",
    "What are common misconceptions about {topic}?",
    "Create a poem about {topic}.",
    "Tell me a fact about {topic}.",
    "Write a joke about {topic}.",
    "Describe {topic} using a sports analogy."
]

# More diverse topics
topics = [
    "blockchain", "gravity", "photosynthesis", "climate change", "AI", "inflation",
    "quantum physics", "machine learning", "nutrition", "vaccination",
    "data privacy", "recycling", "mental health", "electric vehicles",
    "renewable energy", "genetics", "robotics", "economics", "black holes", "meditation"
]

# More fake responses
responses = [
    "It's a process plants use to make food.",
    "A chain of computers that work together.",
    "It makes things fall to the ground.",
    "It affects how much things cost over time.",
    "A hot topic with many opinions!",
    "People often confuse it with something else.",
    "It has changed the world in the last decade.",
    "Used in both science fiction and real tech.",
    "Very relevant in today’s discussions.",
    "Both kids and adults are impacted by it."
]

# Generate a list of Purview-style logs
def generate_logs(n=10):
    logs = []
    now = datetime.utcnow()
    for _ in range(n):
        prompt = random.choice(templates).format(topic=random.choice(topics))
        response = random.choice(responses)
        timestamp = (now - timedelta(seconds=random.randint(0, 10000))).isoformat() + "Z"
        log_entry = {
            "prompt": prompt,
            "response": response,
            "timestamp": timestamp
        }
        logs.append(log_entry)
    return logs

# Create and export logs
json_logs = generate_logs(10)
print(json.dumps(json_logs, indent=2))



wd = WarpDrive()
project_id = wd.get_args("project_id")
pipeline_id = wd.get_args("pipeline_id")
username = wd.get_args("username")
password = wd.get_args("password")
model_a_id = wd.get_args("model_a_id")
model_b_id = wd.get_args("model_b_id")
evaluation_method = wd.get_args("evaluation_method")
api_key = wd.get_args("api_key")
# your code here


url = "https://nimbus-uno-qa.nginx.solyticspartners.com/genai/llm/abtesting"
base_login_url = "https://nimbus-uno-qa.nginx.solyticspartners.com/auth/auth/login/"
token_url = "https://nimbus-uno-qa.nginx.solyticspartners.com/auth/auth/token/"
 
def generate_device_id():
    return f"3214123195-{uuid.uuid4()}-{int(time.time() * 1000)}"
 
Org = "qa"
# your code here
login_headers = {
            "x-tenant-id": Org,
            "x-device-id": generate_device_id()
        }
login_payload = {
            'email': (None, username),
            'password': (None, password),
            'product_domain': (None, "https://nimbus-uno-qa.solyticspartners.com/")
        }
 
res = requests.post(url=base_login_url, files=login_payload, headers=login_headers)
assert res.status_code == 200, f"Invalid status code for login: {res.status_code}, {res.text}"
 
try:
    response_json = res.json()
    #print("Response JSON:", response_json)
    print(f"Login Successful.")
except ValueError:
    print("Response is not valid JSON:")
    print(res.text)
 
# Parse the redirect URL
parsed_url = urllib.parse.urlparse(response_json["redirect_url"])
query_params = urllib.parse.parse_qs(parsed_url.query)
 
# Extract the 'code' parameter
code = query_params.get("code", [None])[0]
session_id = response_json.get("session_id")
#print(session_id)
# print("Extracted Code:", code)
 
token_payload = {"code": code}
token_res = requests.post(url=token_url, headers=login_headers, data=token_payload)
# print(self.token_payload)
assert token_res.status_code == 200, f"Invalid status code for login: {token_res.status_code}, {token_res.text}"
 
try:
    print("Token API Successful.")
    print("Acquiring access token.")
    token_data = token_res.json()  # parse JSON from response
    token = 'Bearer ' + token_data["access_token"]
    print(token)
except Exception as exp:
    print(f"Failed to acquire token \n{exp}\n", token_res.text)
    
    




def convert_json_to_prompt_data(json_data, latency=None):
    """
    Convert a list of dicts with 'prompt', 'response', and 'timestamp' keys
    to prompt_data format with optional latency added.
    """
    return [
        {
            "prompt": entry.get("prompt", ""),
            "response": entry.get("response", ""),
            "timestamp": entry.get("timestamp", ""),
            "latency": latency
        }
        for entry in json_data
    ]

prompt_data = convert_json_to_prompt_data(json_logs, latency=0.25)


def generate_prompt_list(n=5):
    """
    Generate a list of dynamically created prompts only.
    """
    prompts = []
    for _ in range(n):
        topic = random.choice(topics)
        template = random.choice(templates)
        prompt = template.format(topic=topic)
        prompts.append(prompt)
    return prompts

# Create and use prompts list
prompts_list = generate_prompt_list(5)
print(prompts_list)



headers = {
        "Authorization": token,
        "Org": Org,
        "Content-Type": "application/json"
    }


input_cost = random_int = random.randint(10, 20)
output_cost = random_int = random.randint(10, 20)
if evaluation_method == "human_eval":
    payload = {
            "project_id": project_id,
            "pipeline_id": pipeline_id,
            "model_a_id": model_a_id,
            "model_b_id": model_b_id,
            "evaluation_method": evaluation_method,
            "prompts": prompts_list
        }
elif evaluation_method == "openai_eval":
    payload = {
            "project_id": project_id,
            "pipeline_id": pipeline_id,
            "model_a_id": model_a_id,
            "model_b_id": model_b_id,
            "evaluation_method": evaluation_method,
            "prompts": prompts_list,
            "api_key": api_key
        }
    
print(payload)

response = requests.post(url = url, headers=headers, json=payload)

if response.status_code == 200:
    print(response.json())
else:
    response.raise_for_status()



    
    
    



